# Cálculo 2 — Visualizações Interativas

Tópicos de **Cálculo em Várias Variáveis** com foco em visualização geométrica:

1. **Superfícies e Curvas de Nível** — $f(x,y)$ como superfície 3-D e suas curvas de nível
2. **Derivadas Parciais** — $\partial f/\partial x$ e $\partial f/\partial y$ como superfícies
3. **Vetor Gradiente** — campo $\nabla f$ sobreposto às curvas de nível
4. **Polinômio de Taylor** — aproximação quadrática em torno de um ponto
5. **Plano Tangente e Derivada Direcional** — tangência geométrica e variação por direção
6. **Classificação de Pontos Críticos** — máximos, mínimos e pontos de sela via teste da discriminante
7. **Máximos e Mínimos com Restrições** — Multiplicadores de Lagrange e visualização da tangência

> **Como usar:** edite `expr_str` na célula de configuração abaixo e reexecute todas as células.

---

## Como usar este notebook

1. **Troque a função:** edite `expr_str` na célula de configuração (ex.: `"x**2 + y**2"`, `"sin(x)*cos(y)"`, `"x**3 - 3*x*y**2"`).
2. **Reexecute todas as células:** *Kernel → Restart & Run All*.
3. **Ajuste parâmetros locais:** cada seção tem variáveis próprias (ponto de análise, intervalo, ordem de Taylor, raio da restrição) identificadas por comentários `# ===`.
4. **Sintaxe (SymPy):** use `**` para potência, `sin`, `cos`, `exp`, `log`, `sqrt`, `Abs`. A constante $\pi$ é `pi`.

## 0. Configuração

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sympy as sp
from matplotlib import cm
from IPython.display import display

try:
    import plotly.graph_objects as go
    PLOTLY = True
except ImportError:
    PLOTLY = False
    print('plotly nao instalado — execute: pip install plotly')

matplotlib.style.use('ggplot')
%matplotlib inline

# =====================================================================
#  FUNCAO CONFIGURAVEL  ->  edite a string abaixo (variaveis x e y)
# =====================================================================
expr_str = 'x**2 - y**2'
# Outras sugestoes: 'x**2 + y**2', 'sin(x)*cos(y)', 'x**3 - 3*x*y**2',
#                   'exp(-(x**2+y**2))', 'x**4 - 4*x**2 + y**2'
# ---------------------------------------------------------------------

x, y = sp.symbols('x y', real=True)
f_sym  = sp.sympify(expr_str)
f_num  = sp.lambdify((x, y), f_sym, 'numpy')

fx_sym  = sp.diff(f_sym, x)
fy_sym  = sp.diff(f_sym, y)
fxx_sym = sp.diff(f_sym, x, 2)
fyy_sym = sp.diff(f_sym, y, 2)
fxy_sym = sp.diff(sp.diff(f_sym, x), y)
fx_num  = sp.lambdify((x, y), fx_sym,  'numpy')
fy_num  = sp.lambdify((x, y), fy_sym,  'numpy')
fxx_num = sp.lambdify((x, y), fxx_sym, 'numpy')
fyy_num = sp.lambdify((x, y), fyy_sym, 'numpy')
fxy_num = sp.lambdify((x, y), fxy_sym, 'numpy')

XMIN, XMAX, YMIN, YMAX = -3.0, 3.0, -3.0, 3.0


def _grade(xmin=None, xmax=None, ymin=None, ymax=None, n=200):
    xmin = xmin if xmin is not None else XMIN
    xmax = xmax if xmax is not None else XMAX
    ymin = ymin if ymin is not None else YMIN
    ymax = ymax if ymax is not None else YMAX
    Xg, Yg = np.meshgrid(np.linspace(xmin, xmax, n), np.linspace(ymin, ymax, n))
    with np.errstate(divide='ignore', invalid='ignore'):
        Zg = np.asarray(f_num(Xg, Yg), dtype=float)
    Zg[~np.isfinite(Zg)] = np.nan
    return Xg, Yg, Zg


print(f'Funcao configurada: f(x,y) = {expr_str}')
print()
display(sp.Eq(sp.Function('f')(x, y), f_sym))
print('Derivadas parciais:')
display(sp.Eq(sp.Derivative(f_sym, x), fx_sym))
display(sp.Eq(sp.Derivative(f_sym, y), fy_sym))

: 

## 1. Superfícies e Curvas de Nível

Uma função $f:\mathbb{R}^2 \to \mathbb{R}$ define uma **superfície** no espaço 3-D. As **curvas de nível** são as interseções dessa superfície com planos horizontais $z = c$:

$$\{(x,y) \in \mathbb{R}^2 : f(x,y) = c\}$$

Gráfico interativo com `plotly` (arraste para girar) + curvas de nível 2-D.

In [ ]:
# === Parametros ===
n_niveis = 20
# ==================

X, Y, Z = _grade()

fig = plt.figure(figsize=(10, 7), dpi=100, facecolor='w')
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(X, Y, Z, cmap='viridis', alpha=0.88, rstride=3, cstride=3, linewidth=0)
ax.set_title(f'Superficie f(x,y) = {expr_str}', fontsize=12, fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('f(x,y)')
fig.colorbar(surf, ax=ax, shrink=0.45, label='f(x,y)')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 6), dpi=100, facecolor='w')
cf = ax.contourf(X, Y, Z, levels=n_niveis, cmap='viridis', alpha=0.7)
cs = ax.contour( X, Y, Z, levels=n_niveis, colors='k', linewidths=0.6, alpha=0.7)
ax.clabel(cs, inline=True, fontsize=7, fmt='%.1f')
fig.colorbar(cf, ax=ax, label='f(x,y)')
ax.set_title(f'Curvas de nivel de f(x,y) = {expr_str}', fontsize=12, fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

if PLOTLY:
    fig_pl = go.Figure(data=[go.Surface(
        x=X[::3, ::3], y=Y[::3, ::3], z=Z[::3, ::3],
        colorscale='Viridis', opacity=0.92,
    )])
    fig_pl.update_layout(
        title=f'Superficie interativa — f(x,y) = {expr_str} (arraste para girar)',
        scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='f(x,y)',
                   camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))),
        margin=dict(l=0, r=0, b=0, t=45), height=550,
    )
    fig_pl.show()
else:
    print('Instale plotly para visualizacao 3-D interativa: pip install plotly')

## 2. Derivadas Parciais e Diferenciabilidade

A **derivada parcial** mede a taxa de variação de $f$ ao longo de um eixo coordenado:

$$\frac{\partial f}{\partial x}(a,b) = \lim_{h \to 0} \frac{f(a+h,b) - f(a,b)}{h}$$

Geometricamente, $\partial f/\partial x$ é a inclinação da curva obtida fixando $y = b$ (seção paralela ao plano $xz$).

In [ ]:
print('Derivadas parciais de primeira ordem:')
display(sp.Eq(sp.Derivative(f_sym, x), fx_sym))
display(sp.Eq(sp.Derivative(f_sym, y), fy_sym))
print('\nDerivadas parciais de segunda ordem:')
display(sp.Eq(sp.Derivative(f_sym, x, 2), fxx_sym))
display(sp.Eq(sp.Derivative(f_sym, y, 2), fyy_sym))
display(sp.Eq(sp.Derivative(sp.Derivative(f_sym, x), y), fxy_sym))

X, Y, _ = _grade()
with np.errstate(divide='ignore', invalid='ignore'):
    Zx = np.asarray(fx_num(X, Y), dtype=float)
    Zy = np.asarray(fy_num(X, Y), dtype=float)
Zx[~np.isfinite(Zx)] = np.nan
Zy[~np.isfinite(Zy)] = np.nan

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=100, facecolor='w',
                          subplot_kw={'projection': '3d'})
for ax, Zp, titulo, cmap_ in zip(
    axes,
    [Zx, Zy],
    ['df/dx  (derivada em x)', 'df/dy  (derivada em y)'],
    ['plasma', 'inferno']
):
    ax.plot_surface(X, Y, Zp, cmap=cmap_, alpha=0.88, rstride=3, cstride=3, linewidth=0)
    ax.set_title(titulo, fontsize=11, fontweight='bold')
    ax.set_xlabel('x'); ax.set_ylabel('y')

plt.suptitle(f'Derivadas parciais de f(x,y) = {expr_str}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Vetor Gradiente e Suas Aplicações

O **gradiente** reúne as derivadas parciais em um vetor que aponta na **direção de maior crescimento** de $f$:

$$\nabla f(x,y) = \left(\frac{\partial f}{\partial x},\; \frac{\partial f}{\partial y}\right)$$

$\nabla f$ é sempre **perpendicular** às curvas de nível e $|\nabla f|$ é a **taxa máxima de variação** local.

In [ ]:
print('Gradiente:')
display(sp.Matrix([fx_sym, fy_sym]).T)

X, Y, Z = _grade()

Xq, Yq = np.meshgrid(np.linspace(XMIN, XMAX, 18), np.linspace(YMIN, YMAX, 18))
with np.errstate(divide='ignore', invalid='ignore'):
    Gx = np.asarray(fx_num(Xq, Yq), dtype=float)
    Gy = np.asarray(fy_num(Xq, Yq), dtype=float)
Gx[~np.isfinite(Gx)] = 0.0
Gy[~np.isfinite(Gy)] = 0.0

norma = np.sqrt(Gx**2 + Gy**2)
norma[norma < 1e-12] = 1.0
Gxn, Gyn = Gx / norma, Gy / norma

fig, ax = plt.subplots(figsize=(8, 7), dpi=100, facecolor='w')
cf = ax.contourf(X, Y, Z, levels=25, cmap='viridis', alpha=0.55)
cs = ax.contour( X, Y, Z, levels=25, colors='k', linewidths=0.5, alpha=0.6)
ax.clabel(cs, inline=True, fontsize=7, fmt='%.1f')
q = ax.quiver(Xq, Yq, Gxn, Gyn, np.sqrt(Gx**2 + Gy**2),
              cmap='hot_r', scale=22, width=0.004, alpha=0.9)
fig.colorbar(q, ax=ax, label='|nabla f|')
fig.colorbar(cf, ax=ax, label='f(x,y)')
ax.set_title(f'Gradiente e curvas de nivel — f(x,y) = {expr_str}', fontsize=11, fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 4. Polinômio de Taylor

O **Polinômio de Taylor de 2ª ordem** em torno de $(a, b)$:

$$T_2(x,y) = f(a,b) + f_x(a,b)(x-a) + f_y(a,b)(y-b) + \frac{1}{2}\left[f_{xx}(a,b)(x-a)^2 + 2f_{xy}(a,b)(x-a)(y-b) + f_{yy}(a,b)(y-b)^2\right]$$

O gráfico compara $f$, $T_2$ e o erro $|f - T_2|$ lado a lado.

In [ ]:
# === Parametros ===
a_t, b_t = 1.0, 1.0
raio_t   = 2.0
# ==================

sub = {x: a_t, y: b_t}
fa   = float(f_sym.subs(sub))
fxa  = float(fx_sym.subs(sub))
fya  = float(fy_sym.subs(sub))
fxxa = float(fxx_sym.subs(sub))
fyya = float(fyy_sym.subs(sub))
fxya = float(fxy_sym.subs(sub))

print(f'Expansao de Taylor de 2a ordem em ({a_t}, {b_t}):')
print(f'  f(a,b) = {fa:.4f},  df/dx = {fxa:.4f},  df/dy = {fya:.4f}')
print(f'  d2f/dx2 = {fxxa:.4f},  d2f/dy2 = {fyya:.4f},  d2f/dxdy = {fxya:.4f}')


def taylor2(Xi, Yi):
    dx = Xi - a_t; dy = Yi - b_t
    return (fa + fxa*dx + fya*dy
            + 0.5*(fxxa*dx**2 + 2*fxya*dx*dy + fyya*dy**2))


Xt, Yt, Zt = _grade(a_t - raio_t, a_t + raio_t, b_t - raio_t, b_t + raio_t, n=150)
Tt = taylor2(Xt, Yt)
Err = np.abs(Zt - Tt)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), dpi=100, facecolor='w',
                          subplot_kw={'projection': '3d'})
for ax, Zp, titulo, cmap_ in zip(
    axes,
    [Zt, Tt, Err],
    [f'f(x,y)', 'T2 — Taylor 2a ordem', '|f - T2|  (erro)'],
    ['viridis', 'plasma', 'Reds']
):
    ax.plot_surface(Xt, Yt, Zp, cmap=cmap_, alpha=0.88, rstride=3, cstride=3, linewidth=0)
    ax.scatter([a_t], [b_t], [fa], color='red', s=80, zorder=5)
    ax.set_title(titulo, fontsize=10, fontweight='bold')
    ax.set_xlabel('x'); ax.set_ylabel('y')

plt.suptitle(f'Aproximacao de Taylor em ({a_t}, {b_t})', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Plano Tangente e Derivada Direcional

O **plano tangente** à superfície $z = f(x,y)$ no ponto $(a, b, f(a,b))$:

$$z = f(a,b) + f_x(a,b)\,(x-a) + f_y(a,b)\,(y-b)$$

A **derivada direcional** na direção $\mathbf{u} = (\cos\theta, \sin\theta)$:

$$D_{\mathbf{u}}f(a,b) = \nabla f(a,b) \cdot \mathbf{u} = f_x\cos\theta + f_y\sin\theta$$

O máximo é $|\nabla f(a,b)|$, atingido quando $\theta$ aponta na direção do gradiente.

In [ ]:
# === Parametros ===
a_p, b_p = 1.0, 1.0
raio_p   = 2.0
# ==================

sub_p = {x: a_p, y: b_p}
fa_p  = float(f_sym.subs(sub_p))
fxa_p = float(fx_sym.subs(sub_p))
fya_p = float(fy_sym.subs(sub_p))

print(f'Ponto ({a_p}, {b_p}):  f = {fa_p:.4f},  df/dx = {fxa_p:.4f},  df/dy = {fya_p:.4f}')
print(f'Plano tangente: z = {fa_p:.3f} + {fxa_p:.3f}*(x-{a_p}) + {fya_p:.3f}*(y-{b_p})')

Xp, Yp, Zp_f = _grade(a_p - raio_p, a_p + raio_p, b_p - raio_p, b_p + raio_p, n=100)
Zp_tan = fa_p + fxa_p*(Xp - a_p) + fya_p*(Yp - b_p)

fig = plt.figure(figsize=(10, 7), dpi=100, facecolor='w')
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(Xp, Yp, Zp_f,   cmap='viridis', alpha=0.65, rstride=3, cstride=3, linewidth=0)
ax.plot_surface(Xp, Yp, Zp_tan, alpha=0.40, color='orange', rstride=3, cstride=3, linewidth=0)
ax.scatter([a_p], [b_p], [fa_p], color='red', s=100, zorder=5)
ax.set_title(f'Plano tangente em ({a_p}, {b_p})', fontsize=12, fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
plt.tight_layout()
plt.show()

thetas    = np.linspace(0, 2*np.pi, 360)
Df        = fxa_p*np.cos(thetas) + fya_p*np.sin(thetas)
grad_mag  = np.sqrt(fxa_p**2 + fya_p**2)
theta_max = np.arctan2(fya_p, fxa_p)

fig, ax = plt.subplots(figsize=(9, 4), dpi=100, facecolor='w')
ax.plot(np.degrees(thetas), Df, color='#1f77b4', linewidth=2, label='D_u f')
ax.axhline(y= grad_mag, color='red',   linestyle='--', label=f'maximo = {grad_mag:.3f}')
ax.axhline(y=-grad_mag, color='green', linestyle='--', label=f'minimo = {-grad_mag:.3f}')
ax.axvline(x=np.degrees(theta_max), color='red', linestyle=':', alpha=0.7,
           label=f'theta_max = {np.degrees(theta_max):.1f} graus')
ax.set_xlabel('Direcao theta (graus)')
ax.set_ylabel('Derivada direcional D_u f')
ax.set_title('Derivada direcional vs. direcao theta', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.5)
ax.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.show()

## 6. Classificação de Pontos Críticos em Abertos de $\mathbb{R}^2$

Um **ponto crítico** satisfaz $\nabla f = \mathbf{0}$. A **discriminante** classifica o tipo:

$$D = f_{xx}\,f_{yy} - f_{xy}^2$$

| Condição | Tipo |
|----------|------|
| $D > 0$ e $f_{xx} < 0$ | Máximo local |
| $D > 0$ e $f_{xx} > 0$ | Mínimo local |
| $D < 0$ | Ponto de sela |
| $D = 0$ | Inconclusivo |

In [ ]:
D_sym = fxx_sym*fyy_sym - fxy_sym**2

print('Discriminante D = fxx*fyy - fxy^2:')
display(sp.Eq(sp.Symbol('D'), D_sym))

print('\nResolvendo nabla f = 0 ...')
pts = sp.solve([fx_sym, fy_sym], [x, y], dict=True)
print(f'Pontos criticos encontrados: {len(pts)}')

resultados = []
for pt in pts:
    x0 = pt.get(x, sp.Integer(0))
    y0 = pt.get(y, sp.Integer(0))
    try:
        x0v = float(x0.evalf())
        y0v = float(y0.evalf())
    except (TypeError, AttributeError):
        continue
    if not (np.isfinite(x0v) and np.isfinite(y0v)):
        continue
    D_v   = float(D_sym.subs(pt).evalf())
    fxx_v = float(fxx_sym.subs(pt).evalf())
    fval  = float(f_sym.subs(pt).evalf())
    if   D_v > 0 and fxx_v < 0: tipo = 'MAXIMO LOCAL';  cor = 'blue'
    elif D_v > 0 and fxx_v > 0: tipo = 'MINIMO LOCAL';  cor = 'green'
    elif D_v < 0:                tipo = 'PONTO DE SELA'; cor = 'red'
    else:                        tipo = 'inconclusivo';   cor = 'gray'
    resultados.append((x0v, y0v, fval, D_v, fxx_v, tipo, cor))
    print(f'  ({x0v:.3f}, {y0v:.3f}): f = {fval:.3f}, D = {D_v:.3f}, fxx = {fxx_v:.3f} -> {tipo}')

if not resultados:
    print('  Nenhum ponto critico real encontrado no dominio.')

X, Y, Z = _grade()
fig = plt.figure(figsize=(10, 7), dpi=100, facecolor='w')
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(X, Y, Z, cmap='viridis', alpha=0.55, rstride=3, cstride=3, linewidth=0)
for x0v, y0v, fval, D_v, fxx_v, tipo, cor in resultados:
    ax.scatter([x0v], [y0v], [fval], color=cor, s=140, zorder=5,
               edgecolors='k', linewidths=1, label=tipo)
ax.set_title(f'Pontos criticos de f(x,y) = {expr_str}', fontsize=12, fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('f(x,y)')
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
if by_label:
    ax.legend(by_label.values(), by_label.keys(), fontsize=9)
plt.tight_layout()
plt.show()

## 7. Máximos e Mínimos com Restrições — Multiplicadores de Lagrange

Para otimizar $f(x,y)$ sujeito a $g(x,y) = 0$, as condições necessárias são:

$$\nabla f = \lambda\, \nabla g, \qquad g(x,y) = 0$$

No ponto ótimo, as curvas de nível de $f$ e a curva $g = 0$ são **tangentes** (gradientes paralelos).

> Edite `constr_str` abaixo para trocar a restrição.

In [ ]:
# === Configuracao da restricao ===
constr_str = 'x**2 + y**2 - 4'
# =================================

lam   = sp.Symbol('lambda')
g_sym = sp.sympify(constr_str)
gx    = sp.diff(g_sym, x)
gy    = sp.diff(g_sym, y)
g_num = sp.lambdify((x, y), g_sym, 'numpy')

print('Condicoes de Lagrange:')
display(sp.Eq(sp.Derivative(f_sym, x), lam * gx))
display(sp.Eq(sp.Derivative(f_sym, y), lam * gy))
display(sp.Eq(g_sym, 0))

print('\nResolvendo o sistema ...')
try:
    solucoes = sp.solve(
        [fx_sym - lam*gx, fy_sym - lam*gy, g_sym],
        [x, y, lam], dict=True
    )
except Exception as err:
    solucoes = []
    print(f'  sp.solve falhou: {err}')

print(f'Solucoes encontradas: {len(solucoes)}')
pontos_opt = []
for sol in solucoes:
    try:
        xv   = float(sol[x].evalf())
        yv   = float(sol[y].evalf())
        lamv = float(sol[lam].evalf())
        fv   = float(f_sym.subs(sol).evalf())
        if not (np.isfinite(xv) and np.isfinite(yv)):
            continue
        pontos_opt.append((xv, yv, fv, lamv))
        print(f'  ({xv:.4f}, {yv:.4f}): f = {fv:.4f}, lambda = {lamv:.4f}')
    except (KeyError, TypeError, AttributeError):
        continue

X, Y, Z = _grade()
with np.errstate(divide='ignore', invalid='ignore'):
    G = np.asarray(g_num(X, Y), dtype=float)

fig, ax = plt.subplots(figsize=(8, 7), dpi=100, facecolor='w')
cf = ax.contourf(X, Y, Z, levels=25, cmap='viridis', alpha=0.60)
cs = ax.contour( X, Y, Z, levels=25, colors='k', linewidths=0.5, alpha=0.6)
ax.clabel(cs, inline=True, fontsize=7, fmt='%.1f')
fig.colorbar(cf, ax=ax, label='f(x,y)')
ax.contour(X, Y, G, levels=[0.0], colors='red', linewidths=3)
ax.plot([], [], 'r-', linewidth=3, label=f'restricao: {constr_str} = 0')
for xv, yv, fv, lamv in pontos_opt:
    ax.plot(xv, yv, 'o', markersize=13, markeredgecolor='k', linewidth=2,
            label=f'f({xv:.2f}, {yv:.2f}) = {fv:.3f}')
for xv, yv, fv, lamv in pontos_opt:
    gfx = float(fx_sym.subs({x: xv, y: yv}).evalf())
    gfy = float(fy_sym.subs({x: xv, y: yv}).evalf())
    n_ = np.sqrt(gfx**2 + gfy**2) + 1e-12
    ax.annotate('', xy=(xv + 0.4*gfx/n_, yv + 0.4*gfy/n_), xytext=(xv, yv),
                arrowprops=dict(arrowstyle='->', color='white', lw=2))
ax.set_title(f'Lagrange: f = {expr_str},  g = {constr_str} = 0',
             fontsize=11, fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_aspect('equal')
ax.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.show()